In [29]:
import pandas as pd

In [91]:
league_df = pd.read_csv('2425_pergame.csv') 
kings_df = pd.read_csv('2425_kings.csv')
draft_df = pd.read_csv('merged.csv')
league_df = league_df[(league_df['Rk'] >= 1) & (league_df['Rk'] <= 10)]
league_df

,Rk,Team,G,MP,FG,FGA,FG%,3P,3PA,3P%,...,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
0,1,Cleveland Cavaliers*,82,240.9,44.5,90.8,0.491,15.9,41.5,0.383,...,0.776,11.2,34.2,45.4,28.1,8.2,4.3,13.2,18.1,121.9
1,2,Memphis Grizzlies*,82,240.3,44.8,93.3,0.479,13.9,37.9,0.367,...,0.786,12.9,34.4,47.3,28.4,8.9,5.6,15.7,20.9,121.7
2,3,Denver Nuggets*,82,242.1,45.4,89.8,0.506,12.0,31.9,0.376,...,0.770,11.2,34.5,45.7,31.0,8.0,4.9,14.3,17.6,120.8
3,4,Oklahoma City Thunder*,82,240.3,44.6,92.7,0.482,14.5,38.8,0.374,...,0.819,10.6,34.2,44.8,26.9,10.3,5.7,11.7,19.9,120.5
4,5,Atlanta Hawks,82,241.2,43.4,91.8,0.472,13.5,37.7,0.358,...,0.775,11.9,32.6,44.5,29.6,9.7,5.1,15.5,19.1,118.2
5,6,Chicago Bulls,82,240.9,43.2,92.0,0.470,15.4,42.0,0.367,...,0.809,10.1,35.8,45.9,29.1,7.6,4.7,14.7,17.6,117.8
6,7,Indiana Pacers*,82,242.1,43.6,89.3,0.488,13.2,35.8,0.368,...,0.789,9.2,32.7,41.8,29.2,8.5,5.5,13.2,18.7,117.4
7,8,Boston Celtics*,82,241.8,41.6,90.0,0.462,17.8,48.2,0.368,...,0.799,11.4,33.9,45.3,26.1,7.2,5.5,11.9,15.9,116.3
8,9,New York Knicks*,82,242.4,43.3,89.2,0.486,12.6,34.1,0.369,...,0.800,10.9,31.8,42.6,27.5,8.2,4.0,13.3,17.2,115.8
9,10,Sacramento Kings,82,242.4,43.0,90.1,0.478,12.6,35.2,0.357,...,0.806,11.0,33.2,44.2,26.5,7.6,4.4,13.3,18.9,115.7


In [92]:
# Clean column names: remove % and spaces, lowercase them
# league_df.columns = league_df.columns.str.strip().str.replace('*', '', regex=False) 
# kings_df.columns = kings_df.columns.str.strip().str.replace('*', '', regex=False) 
# Clean column names: remove % and spaces, lowercase them
# Clean column names but keep %
league_df.columns = (
    league_df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
)
kings_df.columns = (
    kings_df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
)

# Example metrics list now matches the DataFrame
metrics = ['fg%', '3p%', 'ft%', 'orb', 'drb', 'trb', 'ast', 'stl', 'blk', 'tov', 'pts']
league_df.columns   

Index(['rk', 'team', 'g', 'mp', 'fg', 'fga', 'fg%', '3p', '3pa', '3p%', '2p',
       '2pa', '2p%', 'ft', 'fta', 'ft%', 'orb', 'drb', 'trb', 'ast', 'stl',
       'blk', 'tov', 'pf', 'pts'],
      dtype='object')

In [93]:
# Find Sacramento Kings row in league table
kings_team_stats = league_df[league_df['team'].str.contains("Sacramento Kings", case=False, na=False)].iloc[0]
kings_team_stats

rk                    10
team    Sacramento Kings
g                     82
mp                 242.4
fg                  43.0
fga                 90.1
fg%                0.478
3p                  12.6
3pa                 35.2
3p%                0.357
2p                  30.5
2pa                 54.9
2p%                0.555
ft                  17.1
fta                 21.2
ft%                0.806
orb                 11.0
drb                 33.2
trb                 44.2
ast                 26.5
stl                  7.6
blk                  4.4
tov                 13.3
pf                  18.9
pts                115.7
Name: 9, dtype: object

In [94]:
league_means = league_df[metrics].mean(numeric_only=True)
league_means

fg%      0.4814
3p%      0.3687
ft%      0.7929
orb     11.0400
drb     33.7300
trb     44.7500
ast     28.2400
stl      8.4200
blk      4.9700
tov     13.6800
pts    118.6100
dtype: float64

In [95]:
# Combine Kings stats and league averages into one DataFrame
comparison = pd.DataFrame({
    'league_avg': league_means,
    'kings': kings_team_stats[league_means.index]  # make sure order matches
})

# Compute differences
comparison['difference'] = comparison['kings'] - comparison['league_avg']

# For metrics where higher is bad (like turnovers), invert the difference
inverse_metrics = ['tov']  # higher turnovers = weakness
comparison['score'] = comparison.apply(
    lambda row: -row['difference'] if row.name not in inverse_metrics else row['difference'],
    axis=1
)

# Normalize to sum to 1 (absolute values)
total = comparison['score'].abs().sum()
comparison['weight'] = (comparison['score'] / total).round(3)

print("\n--- Needs Profile ---\n")
print(comparison[['kings', 'league_avg', 'difference', 'weight']])




--- Needs Profile ---

     kings  league_avg difference  weight
fg%  0.478      0.4814    -0.0034   0.000
3p%  0.357      0.3687    -0.0117   0.002
ft%  0.806      0.7929     0.0131  -0.002
orb   11.0     11.0400      -0.04   0.005
drb   33.2     33.7300      -0.53   0.070
trb   44.2     44.7500      -0.55   0.073
ast   26.5     28.2400      -1.74   0.230
stl    7.6      8.4200      -0.82   0.108
blk    4.4      4.9700      -0.57   0.075
tov   13.3     13.6800      -0.38  -0.050
pts  115.7    118.6100      -2.91   0.385


In [96]:
# Example mapping: draft column -> team metric
column_mapping = {
    'two_points_made': '2p',
    'two_points_attempted': '2pa',
    'three_points_made': '3p',
    'three_points_attempted': '3pa',
    'free_throws_made': 'ft',
    'free_throws_attempted': 'fta',
    'offensive_rebounds': 'orb',
    'defensive_rebounds': 'drb',
    'assists': 'ast',
    'steals': 'stl',
    'blocked_shots': 'blk',
    'turnovers': 'tov',
    'points': 'pts'
}

# Create normalized stats for international players
draft_df_norm = draft_df.copy()

# Convert counts to percentages where needed
draft_df_norm['fg%'] = (draft_df_norm['two_points_made'] + draft_df_norm['three_points_made']) / \
                       (draft_df_norm['two_points_attempted'] + draft_df_norm['three_points_attempted'])

draft_df_norm['2p%'] = draft_df_norm['two_points_made'] / draft_df_norm['two_points_attempted']
draft_df_norm['3p%'] = draft_df_norm['three_points_made'] / draft_df_norm['three_points_attempted']
draft_df_norm['ft%'] = draft_df_norm['free_throws_made'] / draft_df_norm['free_throws_attempted']

# Map other raw counts directly
for col, mapped in column_mapping.items():
    if mapped not in ['fg%', '3p%', '2p%', 'ft%']:
        draft_df_norm[mapped] = draft_df_norm[col]


In [99]:
# Compute total rebounds
draft_df_norm['trb'] = draft_df_norm['orb'] + draft_df_norm['drb']

# Normalize
draft_metrics = comparison.index.tolist()  # ['fg%', '3p%', 'ft%', 'orb', 'drb', 'trb', 'ast', ...]

for metric in draft_metrics:
    draft_df_norm[metric+'_norm'] = draft_df_norm[metric] / league_means[metric]


In [100]:
for metric in draft_metrics:
    draft_df_norm[metric+'_weighted'] = draft_df_norm[metric+'_norm'] * comparison.loc[metric, 'weight']

draft_df_norm['fit_score'] = draft_df_norm[[m+'_weighted' for m in draft_metrics]].sum(axis=1)


In [102]:
draft_df_norm = draft_df_norm.sort_values('fit_score', ascending=False)

top_players = draft_df_norm[['first_name', 'last_name', 'fit_score'] + draft_metrics].head(10)
print(top_players)

     first_name last_name  fit_score       fg%       3p%       ft%  orb  drb  \
3787        Rod    Haslem  13.405932  0.466377  0.319658  0.747604   82  550   
4914       Odom  Herrmann  13.391748  0.511331  0.356557  0.788382  233  767   
3903  Elizabeth   Jimenez  13.313677  0.480032  0.157895  0.804924  134  609   
3901  Elizabeth   Jimenez  13.060749  0.486034  0.071429  0.837438  209  540   
3907  Elizabeth   Jimenez  12.779681  0.443810  0.367403  0.775920   96  684   
4787      Shawn   Lammers  12.696542  0.497723  0.434783  0.787440  209  458   
3929      Ahmad   Rice Iv  12.620261  0.665323       NaN  0.642202  332  760   
4786      Shawn   Lammers  12.276041  0.550898  0.100000  0.767442  276  562   
4552  Guillermo    Ashley  12.263321  0.504412  0.353535  0.828054   32  234   
4070      Jamie     Burns  12.139823  0.543111  0.383562  0.781116  255  570   

       trb  ast  stl  blk  tov   pts  
3787   632  590   69   17  291  1945  
4914  1000  322   83   94  172  1721  
39